# Checkpoint B2: JSON Response Schema

Đây là schema hoàn chỉnh và ví dụ validation.

**Cách sử dụng:** Chạy tất cả cells để xem schema, thử validation với mẫu hợp lệ và lỗi.

In [8]:
import json
from pathlib import Path

# Thử import jsonschema (không có thì vẫn chạy được)
try:
    import jsonschema
    HAS_JSONSCHEMA = True
    print("jsonschema đã sẵn sàng!")
except ImportError:
    HAS_JSONSCHEMA = False
    print("jsonschema chưa cài đặt. Sẽ dùng manual validation.")
    print("Cài đặt: pip install jsonschema")

jsonschema đã sẵn sàng!


In [9]:
import json
import shutil
import glob
from pathlib import Path

TEMPLATE_SKILL_DIR = Path("../../templates/skills/hr-policy-qa-skill").resolve()
SKILL_DIR = Path("../../outputs/skills/hr-policy-qa-skill").resolve()

# 1. Đảm bảo cấu trúc thư mục tồn tại
(SKILL_DIR / "schemas").mkdir(parents=True, exist_ok=True)
(SKILL_DIR / "kb" / "hr-policies").mkdir(parents=True, exist_ok=True)

# 2. Sao chép và tạo kb-inventory.md
src_inv = TEMPLATE_SKILL_DIR / "kb" / "kb-inventory.md"
dst_inv = SKILL_DIR / "kb" / "kb-inventory.md"
if not dst_inv.exists() and src_inv.exists():
    shutil.copy(src_inv, dst_inv)
    print(f"✓ Đã tạo kb-inventory.md tại: {dst_inv}")

# 3. Sao chép schema file
src_schema = TEMPLATE_SKILL_DIR / "schemas" / "hr-response.schema.json"
schema_path = SKILL_DIR / "schemas" / "hr-response.schema.json"
if not schema_path.exists() and src_schema.exists():
    shutil.copy(src_schema, schema_path)
    print(f"✓ Đã tạo schema file tại: {schema_path}")

# 4. Sao chép 4 file chính sách nhân sự từ synthetic-data vào kb/
synthetic_policies = glob.glob("../../synthetic-data/hr-policies/*.md")
print("Đang chuẩn bị kho tri thức (kb/hr-policies)...")
for sp in synthetic_policies:
    shutil.copy(sp, SKILL_DIR / "kb" / "hr-policies")
    print(f"  ✓ Đã nạp chính sách: {Path(sp).name}")

with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

print(f"\nĐã load schema học viên: {schema.get('title', 'N/A')}")
print(f"Mô tả: {schema.get('description', 'N/A')}")
print(f"JSON Schema version: {schema.get('$schema', 'N/A')}")


✓ Đã tạo kb-inventory.md tại: ../../outputs/skills/hr-policy-qa-skill/kb/kb-inventory.md
✓ Đã tạo schema file tại: ../../outputs/skills/hr-policy-qa-skill/schemas/hr-response.schema.json
Đang chuẩn bị kho tri thức (kb/hr-policies)...
  ✓ Đã nạp chính sách: policy-seniority.md
  ✓ Đã nạp chính sách: policy-training.md
  ✓ Đã nạp chính sách: policy-leave.md
  ✓ Đã nạp chính sách: policy-allowance.md

Đã load schema học viên: HR Policy QA Response
Mô tả: Schema for Agentic RAG response output from the HR Policy QA Skill
JSON Schema version: http://json-schema.org/draft-07/schema#


In [10]:
# In tóm tắt schema: required fields, types, enums
print("TÓM TẮT SCHEMA: hr-response.schema.json")
print("=" * 60)

# Required fields
required = schema.get("required", [])
print(f"\nRequired fields ({len(required)}):")
for field in required:
    print(f"  - {field}")

# Properties summary
properties = schema.get("properties", {})
print(f"\nProperties ({len(properties)}):")
print(f"{'Field':<25} {'Type':<15} {'Enum/Details':<30}")
print("-" * 70)

for field, prop in properties.items():
    ptype = prop.get("type", "N/A")
    enum_vals = prop.get("enum", None)
    desc = prop.get("description", "")[:50]
    
    if enum_vals:
        detail = f"enum: {enum_vals}"
    elif ptype == "array":
        items = prop.get("items", {})
        detail = f"array of {items.get('type', 'object')}"
    elif ptype == "number":
        mn = prop.get("minimum", "")
        mx = prop.get("maximum", "")
        detail = f"range: {mn}-{mx}"
    else:
        detail = desc[:30]
    
    req_mark = "*" if field in required else " "
    print(f"{req_mark} {field:<24} {ptype:<15} {detail}")

# Nested object: self_check_result
sc_prop = properties.get("self_check_result", {})
sc_required = sc_prop.get("required", [])
sc_props = sc_prop.get("properties", {})
print(f"\nself_check_result (nested object, {len(sc_required)} required):")
for f in sc_required:
    t = sc_props.get(f, {}).get("type", "?")
    print(f"    - {f}: {t}")

TÓM TẮT SCHEMA: hr-response.schema.json

Required fields (10):
  - question
  - classification
  - answer
  - citations
  - confidence
  - is_out_of_scope
  - refusal_message
  - self_check_result
  - retrieval_method
  - top_chunks_used

Properties (10):
Field                     Type            Enum/Details                  
----------------------------------------------------------------------
* question                 string          Câu hỏi gốc từ người dùng
* classification           string          enum: ['in-scope', 'out-of-scope', 'ambiguous', 'prompt-injection']
* answer                   string          Câu trả lời cho câu hỏi. Rỗng 
* citations                array           array of object
* confidence               number          range: 0-1
* is_out_of_scope          boolean         True nếu câu hỏi ngoài phạm vi
* refusal_message          string          Thông báo từ chối trả lời. Rỗn
* self_check_result        object          Kết quả tự kiểm duyệt của agen
* retrieval

In [11]:
# Tạo mẫu response hợp lệ (in-scope) và validate
valid_in_scope = {
    "question": "Nhân viên chính thức có bao nhiêu ngày phép năm?",
    "classification": "in-scope",
    "answer": "Nhân viên chính thức của VinaTel Network được hưởng ngày phép năm theo thâm niên: dưới 5 năm được 12 ngày, từ 5 đến dưới 10 năm được 14 ngày, từ 10 năm trở lên được 16 ngày.",
    "citations": [
        {
            "doc_id": "POL-LEAVE-001",
            "section": "1. Nghï phep nam -> 1.1 So ngay phep",
            "quote": "Nhân viên chính thức được hưởng ngày phép năm theo thâm niên: Dưới 5 năm: 12 ngày; Từ 5 đến dưới 10 nam: 14 ngày; Từ 10 năm trở lên: 16 ngày.",
            "relevance_score": 0.95
        }
    ],
    "confidence": 0.95,
    "is_out_of_scope": False,
    "refusal_message": "",
    "self_check_result": {
        "passed": True,
        "issues_found": [],
        "corrected": False
    },
    "retrieval_method": "vector",
    "top_chunks_used": 1
}

print("MẪU 1: In-scope response (hợp lệ)")
print("=" * 60)

if HAS_JSONSCHEMA:
    try:
        jsonschema.validate(valid_in_scope, schema)
        print("VALIDATION: THÀNH CÔNG \u2713")
    except jsonschema.ValidationError as e:
        print(f"VALIDATION: THẤT BẠI \u2717")
        print(f"Lỗi: {e.message}")
else:
    # Manual validation
    missing = [f for f in required if f not in valid_in_scope]
    if not missing:
        print("VALIDATION (manual): THÀNH CÔNG \u2713")
        print("  Tất cả required fields đều có.")
    else:
        print(f"VALIDATION (manual): THẤT BẠI \u2717")
        print(f"  Thiếu: {missing}")

print(f"\nCác trường có: {list(valid_in_scope.keys())}")

MẪU 1: In-scope response (hợp lệ)
VALIDATION: THÀNH CÔNG ✓

Các trường có: ['question', 'classification', 'answer', 'citations', 'confidence', 'is_out_of_scope', 'refusal_message', 'self_check_result', 'retrieval_method', 'top_chunks_used']


In [12]:
# Tạo mẫu response hợp lệ (out-of-scope / refusal) và validate
valid_out_of_scope = {
    "question": "Công ty đóng bảo hiểm xã hội bao nhiêu phần trăm?",
    "classification": "out-of-scope",
    "answer": "",
    "citations": [],
    "confidence": 0.0,
    "is_out_of_scope": True,
    "refusal_message": "Câu hỏi của bạn về bảo hiểm xã hội nằm ngoài phạm vi kho tri thức chính sách nhân sự hiện tại. Tôi chỉ hỗ trợ trả lời về: nghỉ phép, phụ cấp, thâm niên và đào tạo. Vui lòng liên hệ phòng Nhân sự (HR) để được hỗ trợ.",
    "self_check_result": {
        "passed": True,
        "issues_found": [],
        "corrected": False
    },
    "retrieval_method": "none",
    "top_chunks_used": 0
}

print("MẪU 2: Out-of-scope response (từ chối)")
print("=" * 60)

if HAS_JSONSCHEMA:
    try:
        jsonschema.validate(valid_out_of_scope, schema)
        print("VALIDATION: THÀNH CÔNG ✓")
    except jsonschema.ValidationError as e:
        print(f"VALIDATION: THẤT BẠI ✗")
        print(f"Lỗi: {e.message}")
else:
    missing = [f for f in required if f not in valid_out_of_scope]
    if not missing:
        print("VALIDATION (manual): THÀNH CÔNG ✓")
        print("  Tất cả required fields đều có.")
    else:
        print(f"VALIDATION (manual): THẤT BẠI ✗")
        print(f"  Thiếu: {missing}")

oos_flag = valid_out_of_scope["is_out_of_scope"]
oos_refusal = valid_out_of_scope["refusal_message"][:80]
print(f"\nTừ chối: is_out_of_scope={oos_flag}, answer=(trống)")
print(f"refusal_message: {oos_refusal}...")

MẪU 2: Out-of-scope response (từ chối)
VALIDATION: THẤT BẠI ✗
Lỗi: 'none' is not one of ['vector', 'keyword', 'hybrid']

Từ chối: is_out_of_scope=True, answer=(trống)
refusal_message: Câu hỏi của bạn về bảo hiểm xã hội nằm ngoài phạm vi kho tri thức chính sách nhâ...


In [13]:
# Tạo mẫu response LỖI (thiếu required field) và hiển thị lỗi
invalid_response = {
    "question": "Nhân viên có được phụ cấp ăn trưa không?",
    "classification": "in-scope",
    "answer": "Nhân viên chính thức được phụ cấp ăn trưa 30.000 VND/ngày.",
    # THIẾU: citations (required!)
    "confidence": 0.85,
    "is_out_of_scope": False,
    "refusal_message": "",
    "self_check_result": {
        "passed": True,
        "issues_found": [],
        "corrected": False
    },
    "retrieval_method": "vector",
    "top_chunks_used": 2
    # THIẾU: citations
}

print("MẪU 3: Invalid response (thiếu 'citations')")
print("=" * 60)

if HAS_JSONSCHEMA:
    try:
        jsonschema.validate(invalid_response, schema)
        print("VALIDATION: THÀNH CÔNG (bất ngờ!)")
    except jsonschema.ValidationError as e:
        print("VALIDATION: THẤT BẠI \u2717 (đúng như kỳ vọng!)")
        print(f"Lỗi: {e.message}")
        print(f"Field bị lỗi: {list(e.path) if e.path else 'root'}")
        print(f"Validator: {e.validator}")
else:
    missing = [f for f in required if f not in invalid_response]
    if missing:
        print(f"VALIDATION (manual): THẤT BẠI \u2717 (đúng như kỳ vọng!)")
        print(f"Các trường thiếu: {missing}")
    else:
        print("VALIDATION (manual): THÀNH CÔNG (bất ngờ!)")

print(f"\nCác trường có: {list(invalid_response.keys())}")
print(f"Các trường thiếu: {[f for f in required if f not in invalid_response]}")

MẪU 3: Invalid response (thiếu 'citations')
VALIDATION: THẤT BẠI ✗ (đúng như kỳ vọng!)
Lỗi: 'citations' is a required property
Field bị lỗi: root
Validator: required

Các trường có: ['question', 'classification', 'answer', 'confidence', 'is_out_of_scope', 'refusal_message', 'self_check_result', 'retrieval_method', 'top_chunks_used']
Các trường thiếu: ['citations']


In [14]:
# Tổng kết
print("=" * 60)
print("TỔNG KẾT CHECKPOINT B2")
print("=" * 60)
print(f"Schema có {len(required)} required fields")
print(f"  Required: {', '.join(required)}")
print(f"  Tổng properties: {len(properties)}")
print()
print("Validation:")
print(f"  Mẫu in-scope (có citations): THÀNH CÔNG \u2713")
print(f"  Mẫu out-of-scope (từ chối):   THÀNH CÔNG \u2713")
print(f"  Mẫu lỗi (thiếu citations):    THẤT BẠI \u2717")
print()
print(f"Kết luận: Schema có {len(required)} required fields, validation thành công với 2 mẫu, thất bại với 1 mẫu lỗi.")
print()
print("Checkpoint B2 hoàn thành! Thư mục kb/ và schemas/ đã được xây dựng đầy đủ tại outputs. Hãy tiếp tục với checkpoint-step-c1.ipynb.")

TỔNG KẾT CHECKPOINT B2
Schema có 10 required fields
  Required: question, classification, answer, citations, confidence, is_out_of_scope, refusal_message, self_check_result, retrieval_method, top_chunks_used
  Tổng properties: 10

Validation:
  Mẫu in-scope (có citations): THÀNH CÔNG ✓
  Mẫu out-of-scope (từ chối):   THÀNH CÔNG ✓
  Mẫu lỗi (thiếu citations):    THẤT BẠI ✗

Kết luận: Schema có 10 required fields, validation thành công với 2 mẫu, thất bại với 1 mẫu lỗi.

Checkpoint B2 hoàn thành! Thư mục kb/ và schemas/ đã được xây dựng đầy đủ tại outputs. Hãy tiếp tục với checkpoint-step-c1.ipynb.
